In [10]:
import os
from pathlib import Path

import pandas as pd
from docx import Document
import PyPDF2
from loguru import logger


def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text


def extract_text_from_docx(file_path):
    doc = Document(file_path)
    text = "\n".join([p.text for p in doc.paragraphs])
    return text


def extract_text_from_excel(file_path):
    df = pd.read_excel(file_path, sheet_name=None)
    text = ""
    for sheet_name, sheet in df.items():
        text += f"Sheet: {sheet_name}\n"
        text += sheet.astype(str).apply(lambda row: ' | '.join(row), axis=1).str.cat(sep="\n")
        text += "\n"
    return text


def extract_text_from_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def collect_documents(folder_path):
    data = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            ext = Path(file_path).suffix.lower()
            try:
                if ext == ".pdf":
                    text = extract_text_from_pdf(file_path)
                elif ext == ".docx":
                    text = extract_text_from_docx(file_path)
                elif ext in [".xls", ".xlsx"]:
                    text = extract_text_from_excel(file_path)
                elif ext == ".txt":
                    text = extract_text_from_txt(file_path)
                else:
                    continue
                
                data.append({
                    "file_path": file_path,
                    "file_name": file,
                    "extension": ext,
                    "content": text
                })
            except Exception as e:
                logger.error(f"{file_path}: {e}")
    return data

folder_path = os.path.abspath("/home/yugoff/Downloads/projects/my/laboration/project-lab-sp/data")
documents_data = collect_documents(folder_path)

output_data = "documents_data.csv"
df = pd.DataFrame(documents_data)
df.to_csv(output_data, index=False, encoding="utf-8")
logger.info(f"Завершили подготовку данных, все храним в {os.path.join(os.getcwd(), output_data)}!")

2026-02-02 16:48:46.242 | INFO     | __main__:<module>:74 - Завершили подготовку данных, все храним в /home/yugoff/Downloads/projects/my/laboration/project-lab-sp/scripts/full_directory/documents_data.csv!
